In [1]:
import pandas as pd

In [2]:
customers_df = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='customers') # Add the path

orders_df = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='orders') # Add the path

transactions_df = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='transactions')

In [3]:
customers_df.head()

,customer_id,customer_name,segment,zip_code,region,country,city,state,contact_number
0,AA-10315,Alex Avila,Consumer,55407,Central,United States,minneapolis,minnesota,582-262-8228
1,AA-10375,Allen Armold,Consumer,94109,West,United States,san francisco,california,505-671-1025
2,AA-10480,Andrew Allen,Consumer,94122,West,United States,san francisco,california,248-294-9683
3,AA-10645,Anna Andreadi,Consumer,78664,Central,United States,round rock,texas,582-282-8675
4,AB-10015,Aaron Bergman,Consumer,10011,East,United States,new york city,new york,505-559-3741


In [4]:
orders_df.head()

,order_id,customer_id,ship_mode,vendor_id,order_status,order_purchase_date,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,CA-2014-101147,NaN,First Class,VEN02,delivered,2017-12-16 22:28:00,2017-12-16 22:38,2017-12-19 20:32,2017-12-27 18:03,2018-01-18
1,CA-2014-101476,SD-20485,First Class,VEN04,delivered,2017-01-23 13:40:00,2017-01-25 02:50,2017-01-26 11:26,2017-01-30 08:42,2017-02-28
2,CA-2014-101602,NaN,First Class,VEN05,delivered,2017-05-17 21:10:00,2017-05-17 21:22,2017-05-19 10:42,2017-05-24 11:07,2017-06-08
3,CA-2014-101931,TS-21370,First Class,VEN03,delivered,2017-10-22 14:25:00,2017-10-22 14:49,2017-10-23 18:50,2017-10-27 21:23,2017-11-10
4,CA-2014-103058,AG-10270,First Class,VEN01,delivered,2018-01-30 11:03:00,2018-01-30 11:15,2018-02-05 15:44,2018-02-21 17:53,2018-03-13


In [5]:
transactions_df.head()

,id,order_id,product_id,sales_amt,qty,discount,profit_amt
0,2698,CA-2014-145317,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,6827,CA-2016-118689,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,8154,CA-2017-140151,OFF-LA-10000240,14.6200,2,0.00,6.8714
3,2624,CA-2017-127180,FUR-TA-10000577,957.5775,5,0.45,-383.0310
4,4191,CA-2017-166709,OFF-ST-10000760,22.3680,2,0.20,2.5164


# customer loyalty tier report

## Filtering data for last 6 months

In [6]:
orders_df["order_purchase_date"] = pd.to_datetime(
    orders_df["order_purchase_date"]
)

In [7]:
latest_date = orders_df["order_purchase_date"].max()
latest_date

Timestamp('2018-08-30 13:07:00')

In [8]:
cutoff_date = latest_date - pd.DateOffset(months=6)
cutoff_date

Timestamp('2018-02-28 13:07:00')

In [9]:
recent_orders = orders_df[
    orders_df["order_purchase_date"] >= cutoff_date
].copy()

In [10]:
recent_orders.head()

,order_id,customer_id,ship_mode,vendor_id,order_status,order_purchase_date,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5,CA-2014-103100,AB-10105,First Class,VEN02,delivered,2018-08-15 09:38:00,2018-08-17 03:10,2018-08-17 15:40,2018-08-22 21:13,2018-09-12
6,CA-2014-103317,DM-13525,First Class,VEN01,delivered,2018-08-06 08:44:00,2018-08-06 09:04,2018-08-07 16:11,2018-08-14 22:18,2018-08-21
7,CA-2014-103366,EH-13990,First Class,VEN02,delivered,2018-04-04 18:42:00,2018-04-04 18:55,2018-04-07 01:12,2018-04-19 00:28,2018-04-25
8,CA-2014-103429,LW-16825,First Class,VEN05,delivered,2018-08-09 14:08:00,2018-08-10 03:15,2018-08-10 14:13,2018-08-21 17:32,2018-08-23
9,CA-2014-103989,MC-17605,First Class,VEN05,shipped,2018-04-18 15:28:00,2018-04-18 15:53,2018-04-23 20:42,NaN,2018-05-18


In [11]:
recent_orders.shape

(2041, 10)

## calculating order amount

In [12]:
order_amount = (
    transactions_df
    .groupby("order_id", as_index=False)["sales_amt"]
    .sum()
    .rename(columns={"sales_amt": "order_amount"})
)

In [13]:
order_amount.head()

,order_id,order_amount
0,CA-2014-100006,82.896
1,CA-2014-100090,2854.700
2,CA-2014-100293,79.360
3,CA-2014-100328,89.544
4,CA-2014-100363,354.000


In [14]:
order_amount.shape

(4957, 2)

## merge with recent orders

In [15]:
recent_orders = recent_orders.merge(order_amount, on="order_id", how="left")

In [16]:
recent_orders.head()

,order_id,customer_id,ship_mode,vendor_id,order_status,order_purchase_date,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_amount
0,CA-2014-103100,AB-10105,First Class,VEN02,delivered,2018-08-15 09:38:00,2018-08-17 03:10,2018-08-17 15:40,2018-08-22 21:13,2018-09-12,42.942
1,CA-2014-103317,DM-13525,First Class,VEN01,delivered,2018-08-06 08:44:00,2018-08-06 09:04,2018-08-07 16:11,2018-08-14 22:18,2018-08-21,575.895
2,CA-2014-103366,EH-13990,First Class,VEN02,delivered,2018-04-04 18:42:00,2018-04-04 18:55,2018-04-07 01:12,2018-04-19 00:28,2018-04-25,3.984
3,CA-2014-103429,LW-16825,First Class,VEN05,delivered,2018-08-09 14:08:00,2018-08-10 03:15,2018-08-10 14:13,2018-08-21 17:32,2018-08-23,1028.466
4,CA-2014-103989,MC-17605,First Class,VEN05,shipped,2018-04-18 15:28:00,2018-04-18 15:53,2018-04-23 20:42,NaN,2018-05-18,195.118


In [17]:
recent_orders.shape

(2041, 11)

In [18]:
recent_orders["order_amount"] = recent_orders["order_amount"].fillna(0)

## customer metrics

In [19]:
customer_metrics = (
    recent_orders
    .groupby("customer_id")
    .agg(
        total_spending=("order_amount", "sum"),
        order_count=("order_id", "nunique")
    )
    .reset_index()
)

In [20]:
customer_metrics.head()

,customer_id,total_spending,order_count
0,AA-10315,2637.518,1
1,AA-10375,29.320,3
2,AA-10480,2235.244,2
3,AA-10645,230.895,2
4,AB-10015,77.144,2


## customer loyalty

In [21]:
customer_loyalty = customer_metrics.merge(
    customers_df[["customer_id", "customer_name"]],
    on="customer_id",
    how="left"
)

In [22]:
customer_loyalty.head()

,customer_id,total_spending,order_count,customer_name
0,AA-10315,2637.518,1,Alex Avila
1,AA-10375,29.320,3,Allen Armold
2,AA-10480,2235.244,2,Andrew Allen
3,AA-10645,230.895,2,Anna Andreadi
4,AB-10015,77.144,2,Aaron Bergman


## function for assigning tier

In [23]:
def assign_tier_and_discount(row):
    spending = row["total_spending"]
    orders = row["order_count"]

    if spending < 500:
        tier = "Silver"
        discount = 4 if orders >= 10 else 2

    elif spending <= 2000:
        tier = "Gold"
        discount = 8 if orders >= 10 else 6

    else:
        tier = "Platinum"
        discount = 15 if orders >= 10 else 10

    return pd.Series([tier, f"{discount}%"])

In [24]:
customer_loyalty[["tier", "discount_applicable"]] = (
    customer_loyalty.apply(assign_tier_and_discount, axis=1)
)

In [25]:
customer_loyalty.head()

,customer_id,total_spending,order_count,customer_name,tier,discount_applicable
0,AA-10315,2637.518,1,Alex Avila,Platinum,10%
1,AA-10375,29.320,3,Allen Armold,Silver,2%
2,AA-10480,2235.244,2,Andrew Allen,Platinum,10%
3,AA-10645,230.895,2,Anna Andreadi,Silver,2%
4,AB-10015,77.144,2,Aaron Bergman,Silver,2%


## customer loyalty tier report

In [26]:
customer_loyalty_tier_report = customer_loyalty[
    [
        "customer_id",
        "customer_name",
        "total_spending",
        "order_count",
        "tier",
        "discount_applicable",
    ]
].sort_values(
    by="total_spending",
    ascending=False
).reset_index(drop=True)

customer_loyalty_tier_report.head()

,customer_id,customer_name,total_spending,order_count,tier,discount_applicable
0,JE-15745,Joel Eaton,12856.724,5,Platinum,10%
1,EB-13840,Ellis Ballard,12110.642,2,Platinum,10%
2,EB-14170,Evan Bailliet,10538.204,4,Platinum,10%
3,ER-13855,Elpida Rittenbach,10284.142,3,Platinum,10%
4,CC-12685,Craig Carroll,10187.050,2,Platinum,10%


# discounted pricing report

In [38]:
new_orders_url = "https://cdn.enqurious.com/documents/517cad90-cfb3-48fe-975e-256d942412ac_neworders.csv"

In [39]:
new_orders_df = pd.read_csv(new_orders_url)

In [40]:
new_orders_df.head()

,customer_id,Order_id,original_price
0,AB-10105,CA-2014-103392,55.462
1,DM-13525,CA-2014-103254,66.374
2,RT-13456,CA-2014-103445,588.678
3,JH-10250,CA-2014-103136,224.731
4,AB-10105,CA-2014-103283,578.572


In [41]:
new_orders_df.columns = new_orders_df.columns.str.lower()

In [42]:
new_orders_df.shape

(20, 3)

In [32]:
discounted_pricing_report = new_orders_df.merge(
    customer_loyalty_tier_report[
        ["customer_id", "customer_name", "discount_applicable"]
    ],
    on="customer_id",
    how="left"
)

In [33]:
discounted_pricing_report["discount_applicable"] = (
    discounted_pricing_report["discount_applicable"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

In [34]:
discounted_pricing_report["discounted_price"] = (
    discounted_pricing_report["original_price"] *
    (1 - discounted_pricing_report["discount_applicable"] / 100)
).round(2)

In [35]:
discounted_pricing_report = discounted_pricing_report[
    [
        "order_id",
        "customer_id",
        "customer_name",
        "original_price",
        "discount_applicable",
        "discounted_price",
    ]
]

In [36]:
discounted_pricing_report.head()

,order_id,customer_id,customer_name,original_price,discount_applicable,discounted_price
0,CA-2014-103392,AB-10105,Adrian Barton,55.462,6.0,52.13
1,CA-2014-103254,DM-13525,Don Miller,66.374,6.0,62.39
2,CA-2014-103445,RT-13456,NaN,588.678,NaN,NaN
3,CA-2014-103136,JH-10250,NaN,224.731,NaN,NaN
4,CA-2014-103283,AB-10105,Adrian Barton,578.572,6.0,543.86


In [37]:
discounted_pricing_report.shape

(20, 6)